# 03 — Embeddings: The Model's Dictionary

The embedding layer is the model's first contact with language. It maps each token ID to a dense vector. This notebook explores what that space looks like after training.

**Key question**: Do similar words end up near each other?

In [ ]:
import sys
sys.path.insert(0, "..")

import torch
import numpy as np
import matplotlib.pyplot as plt
from utils.model_loading import load_tlens_model, get_tokenizer, format_param_count
from utils.constants import COLORS

MODEL_SIZE = "0.5b"
model = load_tlens_model(MODEL_SIZE)
tokenizer = get_tokenizer(MODEL_SIZE)

W_E = model.W_E  # Embedding matrix
print(f"Embedding matrix shape: {list(W_E.shape)}")
print(f"  → {W_E.shape[0]:,} tokens, each mapped to a {W_E.shape[1]}-dimensional vector")
print(f"  → {format_param_count(W_E.numel())} parameters")
print(f"  → Tied with unembedding: {not model.cfg.untied_unembed}")

## Token Similarity Search

Given a token, find its nearest neighbors in embedding space using cosine similarity.

In [ ]:
@torch.no_grad()
def find_nearest(token_str, k=10):
    """Find k nearest tokens by cosine similarity in embedding space."""
    token_ids = model.to_tokens(token_str)[0]
    # Use the last non-BOS token
    token_id = token_ids[-1] if len(token_ids) > 1 else token_ids[0]
    
    query = W_E[token_id].float()
    embeddings = W_E.float()
    
    # Cosine similarity
    sims = torch.nn.functional.cosine_similarity(query.unsqueeze(0), embeddings, dim=1)
    top_k = torch.topk(sims, k + 1)  # +1 because the token itself is included
    
    print(f"Nearest neighbors of '{token_str}' (token_id={token_id.item()}):")
    for i, (sim, idx) in enumerate(zip(top_k.values, top_k.indices)):
        decoded = tokenizer.decode([idx.item()])
        marker = " ←" if idx == token_id else ""
        print(f"  {i+1:>2}. {sim:.4f}  '{decoded}'{marker}")
    print()

# Search for different types of tokens
for token in ["python", "Paris", "happy", " 42", "+", " the"]:
    find_nearest(token)

## Embedding Norms

Not all token embeddings are equal. Some tokens have larger L2 norms than others — this affects how much influence they have on downstream computation.

In [ ]:
norms = W_E.float().norm(dim=1).cpu().numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of norms
ax1.hist(norms, bins=100, color=COLORS["embedding"], alpha=0.8, edgecolor="none")
ax1.set_xlabel("L2 Norm")
ax1.set_ylabel("Count")
ax1.set_title("Distribution of Token Embedding Norms")
ax1.axvline(norms.mean(), color="red", linestyle="--", label=f"Mean: {norms.mean():.2f}")
ax1.legend()

# Top and bottom norm tokens
top_idx = np.argsort(norms)[-10:][::-1]
bottom_idx = np.argsort(norms)[:10]

print("Highest-norm tokens (most influential):")
for idx in top_idx:
    print(f"  norm={norms[idx]:.3f}  '{tokenizer.decode([idx])}'")

print("\nLowest-norm tokens (least influential):")
for idx in bottom_idx:
    print(f"  norm={norms[idx]:.3f}  '{tokenizer.decode([idx])}'")

# Norm vs token ID (position in vocabulary)
ax2.scatter(range(len(norms)), norms, s=0.1, alpha=0.3, color=COLORS["embedding"])
ax2.set_xlabel("Token ID")
ax2.set_ylabel("L2 Norm")
ax2.set_title("Embedding Norm vs Token ID")
plt.tight_layout()
plt.show()

## Dimensional Analysis: What Do Individual Dimensions Encode?

Each of the 896 dimensions in the embedding vector may encode different features. Let's look at which tokens score highest/lowest on specific dimensions.

In [ ]:
@torch.no_grad()
def inspect_dimension(dim_idx, k=10):
    """Show tokens that score highest and lowest on a specific embedding dimension."""
    values = W_E[:, dim_idx].float().cpu()
    top_k = torch.topk(values, k)
    bottom_k = torch.topk(-values, k)
    
    print(f"Dimension {dim_idx}:")
    print(f"  HIGHEST: ", end="")
    print(", ".join([f"'{tokenizer.decode([i])}' ({v:.2f})" for v, i in zip(top_k.values, top_k.indices)]))
    print(f"  LOWEST:  ", end="")
    print(", ".join([f"'{tokenizer.decode([i])}' ({-v:.2f})" for v, i in zip(bottom_k.values, bottom_k.indices)]))
    print()

# Look at a few dimensions — you may find some that encode interpretable features
for dim in [0, 50, 100, 200, 500]:
    inspect_dimension(dim)

## UMAP Visualization: Global Structure

Let's project 5,000 token embeddings to 2D using UMAP to see the global structure of the embedding space.

In [ ]:
from umap import UMAP

# Sample 5000 tokens
n_sample = 5000
torch.manual_seed(42)
indices = torch.randperm(W_E.shape[0])[:n_sample]
embeddings = W_E[indices].float().cpu().numpy()

# Categorize tokens
sample_tokens = [tokenizer.decode([idx.item()]) for idx in indices]
categories = []
for tok in sample_tokens:
    stripped = tok.strip()
    if stripped.isdigit():
        categories.append("Digits")
    elif stripped and all(c in "!@#$%^&*()[]{}|;:',.<>?/\\`~+-=_\"" for c in stripped):
        categories.append("Punctuation")
    elif any('\u4e00' <= c <= '\u9fff' for c in tok):
        categories.append("Chinese")
    elif any(c in "{}()[];" for c in tok) or tok.strip() in ["def", "class", "import", "return", "if", "else", "for", "while"]:
        categories.append("Code")
    else:
        categories.append("Text")

# UMAP projection
print("Running UMAP (this may take ~30 seconds)...")
reducer = UMAP(n_components=2, n_neighbors=30, min_dist=0.3, random_state=42)
projected = reducer.fit_transform(embeddings)
print("Done.")

# Plot
fig, ax = plt.subplots(figsize=(12, 10))
cat_colors = {"Text": "#cccccc", "Digits": "#e41a1c", "Punctuation": "#377eb8",
              "Chinese": "#4daf4a", "Code": "#ff7f00"}
cat_order = ["Text", "Digits", "Punctuation", "Chinese", "Code"]

for cat in cat_order:
    mask = [c == cat for c in categories]
    alpha = 0.15 if cat == "Text" else 0.7
    size = 3 if cat == "Text" else 8
    ax.scatter(projected[mask, 0], projected[mask, 1], c=cat_colors[cat],
               label=f"{cat} ({sum(mask)})", s=size, alpha=alpha)

ax.set_title("UMAP of Token Embeddings (5,000 tokens)")
ax.legend(markerscale=3, loc="upper right")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
plt.tight_layout()
plt.show()

## Tied Embeddings

In the 0.5B model, `W_E == W_U` (the embedding and unembedding matrices are shared). This means the same vector that represents a token as *input* also defines how the model predicts that token as *output*. The 7B model separates these — `W_E ≠ W_U` — allowing the model to learn different representations for input and output contexts.